# TRAIN V3 DISK-SAFE - GROUP D
Train HCM0674, bonsai, and chair to 40k on two T4 GPUs. Attach 01_BASE_CODE.zip, 02_ROUND2_DATA.zip, and 03_ROUND2_PATCH_V3_DISKSAFE.zip. This version avoids writing huge optimizer checkpoints, monitors free disk, cleans each completed scene, and can still resume a checkpoint supplied as an input.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA is not available'
assert torch.cuda.device_count() >= 2, 'Round 2 Group D requires GPU T4 x2'
print('GPU count:', torch.cuda.device_count())

In [ ]:
from pathlib import Path
import shutil
input_root = Path('/kaggle/input')
work_dir = Path('/kaggle/working/vai_nvs')
if work_dir.exists():
    shutil.rmtree(work_dir)
zip_candidates = list(input_root.rglob('01_BASE_CODE.zip'))
if not zip_candidates:
    zip_candidates = list(input_root.rglob('vai_nvs_kaggle_prod_v3.zip'))
if zip_candidates:
    assert len(zip_candidates) == 1, f'Expected one prod_v3 ZIP, got: {zip_candidates}'
    print('Extracting source package:', zip_candidates[0])
    shutil.unpack_archive(str(zip_candidates[0]), str(work_dir))
else:
    candidates = []
    for p in input_root.rglob('gaussian-splatting'):
        parent = p.parent
        is_full_base = (parent / 'scripts' / 'setup_cuda_extensions_linux.sh').exists() and (p / 'submodules' / 'diff-gaussian-rasterization').exists()
        if is_full_base:
            candidates.append(parent)
    assert len(candidates) == 1, 'BASE CODE missing. Add the Kaggle dataset containing 01_BASE_CODE.zip, then Save & Run All again.'
    shutil.copytree(candidates[0], work_dir)
old_data = work_dir / 'work_undistorted'
if old_data.exists():
    shutil.rmtree(old_data)
old_outputs = work_dir / 'outputs'
if old_outputs.exists():
    shutil.rmtree(old_outputs)
assert (work_dir / 'gaussian-splatting').exists()
assert (work_dir / 'scripts' / 'setup_cuda_extensions_linux.sh').exists(), 'Incomplete BASE CODE: setup script missing'
assert (work_dir / 'gaussian-splatting' / 'submodules' / 'diff-gaussian-rasterization').exists(), 'Incomplete BASE CODE: CUDA submodule missing'
print('Source workspace ready; old 8-scene data removed')

In [ ]:
import os
from pathlib import Path
work_dir = Path('/kaggle/working/vai_nvs')
assert work_dir.exists(), 'BASE CODE was not prepared; check the previous cell'
os.chdir(work_dir)
print('Working directory:', Path.cwd())

In [ ]:
!python -m pip install -U pip "setuptools<82" wheel ninja
!pip install tqdm plyfile opencv-python pillow scipy matplotlib

In [ ]:
from pathlib import Path
import shutil
marker_candidates = list(Path('/kaggle/input').rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
if not marker_candidates:
    hotfix_zips = list(Path('/kaggle/input').rglob('03_ROUND2_PATCH_V3_DISKSAFE.zip'))
    assert len(hotfix_zips) == 1, 'Add exactly one 03_ROUND2_PATCH_V3_DISKSAFE.zip as an input'
    hotfix_extract = Path('/kaggle/working/round2_patch')
    if hotfix_extract.exists():
        shutil.rmtree(hotfix_extract)
    shutil.unpack_archive(str(hotfix_zips[0]), str(hotfix_extract))
    marker_candidates = list(hotfix_extract.rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
assert len(marker_candidates) == 1, f'Round 2 patch marker problem: {marker_candidates}'
hotfix_root = marker_candidates[0].parent
patches = [
    'gaussian-splatting/scene/cameras.py',
    'gaussian-splatting/train.py',
    'gaussian-splatting/render.py',
    'scripts/train_round2_group_2gpu.sh',
    'scripts/train_round2_v2_2gpu.sh',
    'tools/prepare_round2_kaggle.py',
    'tools/undistort_simple_radial_dataset.py',
    'tools/prune_colmap_images_to_existing.py',
    'tools/colmap_io.py',
]
for relative in patches:
    source = hotfix_root / relative
    target = Path('/kaggle/working/vai_nvs') / relative
    assert source.exists(), f'Missing hotfix file: {source}'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
print('Round 2 patch applied')

In [ ]:
!python tools/prepare_round2_kaggle.py --group D

In [ ]:
import os
import torch
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
print('CUDA architecture:', os.environ['TORCH_CUDA_ARCH_LIST'])
!bash scripts/setup_cuda_extensions_linux.sh

In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path
subprocess.run(['python', '-m', 'pip', 'cache', 'purge'], check=False)
for scene in ['HCM0674', 'bonsai', 'chair']:
    model_dir = Path(f'outputs/prod_private_{scene}')
    final_candidates = list(Path('/kaggle/input').rglob(f'prod_private_{scene}/point_cloud/iteration_40000/point_cloud.ply'))
    if final_candidates:
        selected = sorted(final_candidates, key=lambda p: (len(str(p)), str(p)))[0]
        target = model_dir / 'point_cloud/iteration_40000/point_cloud.ply'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(selected, target)
        print('RESTORED FINAL', scene, selected)
        continue
    checkpoints = []
    for candidate in Path('/kaggle/input').rglob(f'prod_private_{scene}/chkpnt*.pth'):
        match = re.fullmatch(r'chkpnt(\d+)\.pth', candidate.name)
        if match and int(match.group(1)) < 40000:
            checkpoints.append((int(match.group(1)), candidate))
    if checkpoints:
        iteration, selected = max(checkpoints, key=lambda item: item[0])
        model_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_link = model_dir / selected.name
        checkpoint_link.symlink_to(selected)
        print('RESTORED CHECKPOINT WITHOUT COPYING DISK DATA', scene, iteration, selected)
train_env = os.environ.copy()
train_env['ROUND2_GROUP'] = 'D'
train_env['V2_ITERATIONS'] = '40000'
train_env['V2_ENABLE_CHECKPOINTS'] = '0'
train_env['V2_MIN_DISK_GIB'] = '6'
subprocess.run(['bash', 'scripts/train_round2_v2_2gpu.sh'], check=True, env=train_env)

In [ ]:
from pathlib import Path
for scene in ['HCM0674', 'bonsai', 'chair']:
    ply = Path(f'outputs/prod_private_{scene}/point_cloud/iteration_40000/point_cloud.ply')
    assert ply.exists(), f'Missing final model: {scene}'
    print(scene, 'OK', round(ply.stat().st_size / 1024**2, 1), 'MiB')
print('ROUND 2 V3 DISK-SAFE GROUP D VERIFIED - SAVE THIS NOTEBOOK VERSION AS A KAGGLE DATASET')